# Experiment: episode length x inference context (seed 42)

**Requires a GPU.** This is the run that produced `data/curves_seed42.csv`.
It is included as provenance; you do not need to run it to check the results -
`02_analysis.ipynb` recomputes everything from the committed CSV, on CPU.

**What it does.** For each of 24 binary OpenML tasks (suite 457), fine-tunes
TabPFN 3.5 with episodes of a fixed length, checkpoints at 25 / 50 / 200 gradient
steps, and evaluates each checkpoint across a sweep of inference context lengths.
The un-adapted model is measured at every context as the reference.

**Protocol.** Fold 0 is sealed and never read. The outer training split is cut
80/20 into fit and report; the fit part is cut 70/15/15 into train, Vscore and
Vselect. Metrics are computed on the report split only.

*Body in French. Set the Kaggle secret `TABPFN_TOKEN` and select a GPU
accelerator before running; neither follows a copied notebook.*

---


# Contexte x fine-tuning — le fine-tuning est-il un substitut du contexte ?**La question.** Tes trois études de fine-tuning ont toutes mesuré à **1024 lignes de contexte**. Le dernier run montre que 1024 est déjà proche de la saturation pour la plupart de tes datasets : le gain moyen est de +0,0124 de log-loss par doublement entre 256 et 2048, et le contexte complet apporte +0,0234. Autrement dit, **« le fine-tuning ne marche pas » a peut-être été mesuré du mauvais côté d'un arbitrage**.L'hypothèse testée ici est simple et intuitive : *le fine-tuning et le contexte fournissent la même information au modèle.* Le contexte la fournit par l'attention, le fine-tuning par les poids. S'ils sont substituables, alors le gain du fine-tuning doit être **une fonction décroissante de la taille du contexte**, positif quand le contexte est famélique, nul ou négatif quand il est abondant. C'est une prédiction de forme, falsifiable, et aucun de tes runs n'a pu la tester puisqu'ils n'ont qu'un seul point sur cet axe.**L'économie du dispositif, qui rend l'expérience possible en une session.** Le fine-tuning **ne dépend pas du contexte d'inférence** : on entraîne une fois par dataset, on enregistre des jalons de poids, et **chaque jalon est ensuite évalué à toutes les tailles de contexte**. On obtient une grille en deux dimensions pour le prix d'une colonne.**Les trois axes balayés.**| Axe | Valeurs | Pourquoi ||:--|:--|:--|| Contexte d'inférence | 32, 64, 128, 256, 512, 1024, 2048, tout le train | on descend **sous** 256, là où le fine-tuning gagnerait s'il gagne un jour || Budget de fine-tuning | 0, 25, 50, 200 pas | 50 est exactement le réglage de tes runs précédents : les anciens chiffres restent lisibles dans les nouveaux || Taille de l'épisode d'entraînement | 128 lignes, et 640 en phase C | réponse à l'objection de relecture décrite plus bas |**L'objection de relecture, et la phase qui y répond.** Les épisodes de fine-tuning font 128 lignes (102 de contexte, 26 de requêtes). Un relecteur dira : « votre fine-tuning spécialise le modèle pour des contextes de 100 lignes, il est normal qu'il n'aide qu'à 100 lignes ». C'est juste, et c'est un *mécanisme*, pas un artefact — mais il faut le montrer. La **phase C** réentraîne avec des épisodes de 640 lignes sur les datasets les plus grands. Si le pic de gain se déplace avec la taille de l'épisode, on tient l'explication ; s'il ne bouge pas, l'effet est bien celui du contexte d'inférence.**Ce que ce run peut donner, et les deux issues sont publiables.**- *Croisement observé* : le fine-tuning gagne à petit contexte, perd à grand. Tes trois négatifs deviennent alors un **résultat conditionnel** : le fine-tuning de TabPFN est redondant dès que le contexte suffit. C'est une affirmation utile et actionnable.- *Pas de croisement* : le fine-tuning est négatif partout, même à 32 lignes. Le négatif devient alors beaucoup **plus fort** qu'avant, parce qu'il survit au régime qui lui était le plus favorable. C'est le résultat que je considère le plus probable, et il faut l'écrire sans le déguiser.**Ce qui est scellé.** Fold extérieur 0 d'OpenML jamais lu. Aucun hyperparamètre n'est choisi sur une validation : le budget de pas est fixé d'avance, il n'y a pas d'arrêt anticipé, pas de sélection de jalon sur une métrique. Le rapport est donc lu une seule fois par (dataset, bras, taille de contexte).**Coût : environ 5 h sur T4**, plus la phase C. Une garde de temps arrête proprement chaque phase et écrit les résultats : le notebook ne meurt jamais sans rien produire.

In [ ]:

import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','tabpfn==9.0.0',
                'openml>=0.15,<0.17','scikit-learn>=1.5,<1.8','scipy','matplotlib','nbformat>=5'],check=True)
print('Dépendances installées. Redémarrer le noyau si Kaggle le demande, puis relancer tout.')

In [ ]:

import os
os.environ['TABPFN_NO_BROWSER']='1'
if not os.environ.get('TABPFN_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('TABPFN_TOKEN')
    except Exception as exc:
        raise RuntimeError('Créer et autoriser le secret Kaggle TABPFN_TOKEN (clé API Prior Labs).') from exc
    if not token: raise RuntimeError('Le secret TABPFN_TOKEN est vide.')
    os.environ['TABPFN_TOKEN']=token.strip(); del token
print('Authentification chargée en mémoire, clé masquée.')

## 1. ConfigurationTout se décide ici, **avant** de regarder le moindre résultat. Le `RUN_HASH` interdit de mélanger deux configurations dans un même dossier.

In [ ]:

from pathlib import Path

SEED = 42
SUITE_ID = 457
ALL_TASK_IDS = [363621,363626,363619,363623,363682,363684,
                363613,363618,363624,363627,363629,363632,
                363671,363674,363676,363679,363681,363689,
                363691,363694,363696,363700,363706,363712]

# Découpage optionnel en sessions. 1 shard = tout en un run.
N_SHARDS, SHARD_ID = 1, 0

# ---- axe 1 : le contexte d'inférence -------------------------------------
# On descend jusqu'à 32 lignes : c'est là que le fine-tuning gagnerait s'il gagne.
# None = tout le train de la cible.
CONTEXT_SIZES     = (32,64,128,256,512,1024,2048,None)
REF_CONTEXT       = 1024      # le point de tes trois runs précédents
SMALL_CONTEXT_MAX = 256       # au-dessous, le tirage du contexte est bruyant
REPEATS_SMALL     = 3         # donc on le répète et on moyenne les métriques

# ---- axe 2 : le budget de fine-tuning ------------------------------------
# Un seul entraînement par dataset, jalonné : les trois budgets ne coûtent
# que l'entraînement le plus long. 50 = le réglage exact de tes runs passés.
FT_MILESTONES = (25,50,200)
FT_CONTEXT, FT_QUERY = 102, 26            # épisode de 128 lignes, ratio 0.2
LEARNING_RATE, WEIGHT_DECAY, GRAD_CLIP = 1e-5, 0.01, 1.0

# ---- axe 3 (phase C) : la taille de l'épisode d'entraînement -------------
# Réponse à l'objection « votre fine-tuning est spécialisé pour 100 lignes ».
RUN_LONG_EPISODE_PHASE = True
LONG_FT_CONTEXT, LONG_FT_QUERY = 512, 128  # épisode de 640 lignes
LONG_FT_MILESTONES = FT_MILESTONES         # MÊMES budgets, sinon rien n'est comparable
LONG_PHASE_TASKS = 4                       # les 4 plus grands trains

# ---- évaluation ----------------------------------------------------------
REPORT_EVAL_CAP = 1024        # requêtes du rapport scellé, identiques pour tous les bras
PRED_BATCHES    = (64,16,4,1)

# ---- garde de temps ------------------------------------------------------
# Chaque phase vérifie le temps écoulé AVANT de commencer un nouveau dataset.
# Un run trop long s'arrête proprement et écrit ce qu'il a : jamais de perte sèche.
TIME_BUDGET_HOURS = 8.0

# OpenML renvoie regulierement des 503 : on reessaie au lieu de perdre le run.
OPENML_RETRIES, OPENML_BACKOFF = 6, 10

SKIP_ON_ERROR = True
MIN_TARGETS = 12
MAX_DATASET_ROWS, MAX_FEATURES = 50_000, 110
MIN_FREE_GIB = 3.0
LOCAL_TRAIN_CAP = REF_CONTEXT   # nom attendu par le code de splits réutilisé

RUN_TAG = 'contexte_x_finetuning_v1'
ON_KAGGLE = Path('/kaggle/working').exists()
ROOT    = (Path('/kaggle/working') if ON_KAGGLE else Path('/mnt/data'))/RUN_TAG   # CSV, conservés
SCRATCH = (Path('/kaggle/temp')   if ON_KAGGLE else Path('/mnt/data'))/RUN_TAG    # poids, éphémère
ROOT.mkdir(parents=True,exist_ok=True); SCRATCH.mkdir(parents=True,exist_ok=True)

SHARD_IDS = ALL_TASK_IDS[SHARD_ID::N_SHARDS]
assert len(ALL_TASK_IDS)==len(set(ALL_TASK_IDS))==24
assert 0<=SHARD_ID<N_SHARDS and SHARD_IDS
assert REF_CONTEXT in CONTEXT_SIZES and None in CONTEXT_SIZES
assert list(FT_MILESTONES)==sorted(set(FT_MILESTONES)) and min(FT_MILESTONES)>=1
assert list(LONG_FT_MILESTONES)==sorted(set(LONG_FT_MILESTONES))
assert all(c is None or c>=8 for c in CONTEXT_SIZES)

print('Résultats :',ROOT,'| poids temporaires :',SCRATCH)
print('Contextes  :',CONTEXT_SIZES,'(None = tout le train)')
print('Budgets FT :',(0,)+tuple(FT_MILESTONES),'pas | épisode de',FT_CONTEXT+FT_QUERY,'lignes')
print('Phase C    :',('ON, épisodes de %d lignes sur %d datasets'%(LONG_FT_CONTEXT+LONG_FT_QUERY,LONG_PHASE_TASKS))
      if RUN_LONG_EPISODE_PHASE else 'OFF')
print(f'Garde de temps : {TIME_BUDGET_HOURS} h')
print('Ordre de grandeur attendu sur T4 : phase A ~0,6 h, phase B ~4,5 h, phase C ~1,5 h.')
print('La phase C est en dernier et se coupe d elle-même : un résultat partiel y reste '
      'lisible, puisque la comparaison est appariée dataset par dataset.')

## 2. Imports et utilitairesIdentiques aux notebooks précédents, **machinerie de fine-tuning comprise** (les trois derniers runs n'en avaient pas besoin ; celui-ci si). Les APIs internes sont épinglées à `tabpfn==9.0.0` : toute montée de version demande de revalider ces assertions.

In [ ]:

import re, gc, json, time, random, shutil, hashlib, inspect, traceback, importlib.metadata
import numpy as np, pandas as pd, torch, openml
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from scipy.stats import spearmanr
from IPython.display import display

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
from tabpfn.base import get_embeddings as tabpfn_get_embeddings
from tabpfn.model_loading import save_tabpfn_model
from tabpfn.finetuning.finetuned_classifier import FinetunedTabPFNClassifier
from tabpfn.finetuning.finetuned_base import _parameters_unused_by_task
from tabpfn.finetuning.data_util import get_preprocessed_dataset_chunks, meta_dataset_collator
from tabpfn.finetuning._torch_compat import GradScaler, autocast, sdpa_kernel_context
from tabpfn.architectures.interface import PerformanceOptions

assert importlib.metadata.version('tabpfn')=='9.0.0', 'tabpfn != 9.0.0 : redémarrer le noyau après pip.'
assert torch.cuda.is_available(), 'Activer le GPU (Settings -> Accelerator -> GPU).'
assert hasattr(ModelVersion,'V3_5')
assert 'shuffle' in inspect.signature(get_preprocessed_dataset_chunks).parameters
DEVICE='cuda:0'; GIB=2**30
print('GPU :',torch.cuda.get_device_name(0),'/',round(torch.cuda.get_device_properties(0).total_memory/GIB,2),'Gio')

def clean_gpu():
    gc.collect(); torch.cuda.empty_cache()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sha(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()

def sha_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''): h.update(chunk)
    return h.hexdigest()

def write_json(path,obj):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    tmp.replace(p)

def write_csv(path,frame):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.tmp'); frame.to_csv(tmp,index=False); tmp.replace(p)

def check_disk(label,where=None,extra=0.0):
    free=shutil.disk_usage(where or SCRATCH).free/GIB
    if free < MIN_FREE_GIB+extra:
        raise RuntimeError(f'{label} : {free:.1f} Gio libres. Exporter les CSV puis relancer un shard plus petit.')

def choose_indices(y,n,seed):
    # Sous-échantillonnage stratifié déterministe, >= 2 exemples par classe.
    y=np.asarray(y); n=min(int(n),len(y))
    classes,counts=np.unique(y,return_counts=True)
    if len(classes)!=2 or min(counts)<2: raise ValueError('Deux classes avec >= 2 lignes requises')
    if n==len(y): return np.arange(len(y))
    if n<4: raise ValueError('Échantillon trop petit')
    k1=max(2,min(int(counts[1]),int(round(n*counts[1]/len(y))))); k0=n-k1
    if k0>counts[0]: k0=int(counts[0]); k1=n-k0
    if k0<2: k0=2; k1=n-k0
    if k1>counts[1] or k0>counts[0] or min(k0,k1)<2: raise ValueError('Stratification impossible')
    rng=np.random.default_rng(int(seed))
    a=rng.choice(np.flatnonzero(y==classes[0]),k0,replace=False)
    b=rng.choice(np.flatnonzero(y==classes[1]),k1,replace=False)
    return np.sort(np.r_[a,b])

def scores(y,p):
    p=np.asarray(p,dtype=np.float64).reshape(-1)
    if not np.isfinite(p).all() or (p<0).any() or (p>1).any(): raise ValueError('Probabilités invalides')
    if len(np.unique(y))!=2: raise ValueError('Métrique indéfinie sur un split mono-classe')
    return dict(auc=float(roc_auc_score(y,p)),
                logloss=float(log_loss(y,np.clip(p,1e-7,1-1e-7),labels=[0,1])),
                ap=float(average_precision_score(y,p)))

def is_oom(e):
    return isinstance(e,torch.cuda.OutOfMemoryError) or 'out of memory' in str(e).lower()

FAILURES=[]
def record_failure(phase,tid,name,exc,**extra):
    # Un échec est écarté du run, jamais masqué : il apparaît dans failures.csv et à l écran.
    row=dict(phase=phase,task_id=tid,name=name,error=type(exc).__name__,
             message=str(exc)[:400],**extra)
    FAILURES.append(row)
    p=ROOT/'failures.csv'
    tmp=p.with_name(p.name+'.tmp'); pd.DataFrame(FAILURES).to_csv(tmp,index=False); tmp.replace(p)
    print(f'  !! ÉCARTÉ [{phase}] {name} ({tid}) — {type(exc).__name__}: {str(exc)[:160]}',flush=True)
    if not SKIP_ON_ERROR: raise exc
    return row

def openml_retry(fn,*a,**kw):
    # OpenML renvoie regulierement des 503, ou coupe en cours de route. On reessaie
    # avec attente croissante plutot que de perdre un run de plusieurs heures.
    last=None
    for k in range(OPENML_RETRIES):
        try: return fn(*a,**kw)
        except Exception as exc:
            last=exc
            if k==OPENML_RETRIES-1: break
            wait=min(OPENML_BACKOFF*(2**k),120)
            print(f'    OpenML indisponible ({type(exc).__name__}) - nouvelle tentative '
                  f'dans {wait}s [{k+1}/{OPENML_RETRIES-1}]',flush=True)
            time.sleep(wait)
    raise last

set_seed(SEED)
print('Utilitaires prêts.')

## 3. Métadonnées et splitsMêmes splits que partout : fold extérieur 0 scellé, fit 80 % / rapport 20 %, puis train 70 % / Vscore 15 % / Vselect 15 %. Vscore et Vselect ne servent pas — aucun hyperparamètre n'est choisi — mais le découpage reste identique pour que les chiffres soient directement comparables à tes quatre runs précédents.

In [ ]:

openml.config.set_root_cache_directory(str(SCRATCH/'openml_cache'))
check_disk('métadonnées OpenML')
# Appartenance a la suite : un controle utile, mais pas au prix du run.
# Les 24 identifiants sont figes et deja valides dans les runs precedents.
try:
    suite=openml_retry(openml.study.get_suite,SUITE_ID)
    allowed=set(map(int,suite.tasks))
    assert set(ALL_TASK_IDS)<=allowed
    print('Suite 457 verifiee.')
except Exception as exc:
    print(f'Suite 457 inverifiable ({type(exc).__name__}) - on poursuit avec les '
          '24 identifiants figes.',flush=True)

def clean_description(s):
    s=re.sub(r'<[^>]+>',' ',str(s or ''))
    s=re.sub(r'https?://\S+',' ',s)
    return re.sub(r'\s+',' ',s).strip()[:2500]

META={}; TASKS={}; TEXT={}; seen=set()
for tid in list(ALL_TASK_IDS):
    try:
        t=openml_retry(openml.tasks.get_task,tid,download_data=False,download_qualities=False)
        ds=openml_retry(t.get_dataset,download_data=False)
        did=int(t.dataset_id)
        if did in seen: raise ValueError(f'{tid}: doublon dataset_id OpenML')
        txt=f'{ds.name}. Prediction target {t.target_name}. '+clean_description(ds.description)
        if len(txt)<20: raise ValueError(f'{tid}: description trop courte pour la baseline textuelle')
        seen.add(did)
        META[tid]={'task_id':tid,'dataset_id':did,'name':str(ds.name),'target':str(t.target_name),
                   'description':txt,'url':f'https://www.openml.org/t/{tid}'}
        TASKS[tid]=t; TEXT[tid]=txt
        print(tid,META[tid]['name'])
    except Exception as exc:
        record_failure('metadata',tid,f'task_{tid}',exc)
ALL_TASK_IDS=[t for t in ALL_TASK_IDS if t in TASKS]
if len(ALL_TASK_IDS)<MIN_TARGETS:
    raise RuntimeError(f'{len(ALL_TASK_IDS)} tâches seulement : voir failures.csv.')
print(f'{len(ALL_TASK_IDS)}/24 tâches OpenML disponibles.')
write_json(ROOT/'catalog_meta.json',META)

In [ ]:

def encode_task(frames):
    # frames[0] = train : c'est le seul sur lequel le prétraitement est ajusté.
    frames=[f.copy().reset_index(drop=True) for f in frames]
    good=[c for c in frames[0].columns
          if frames[0][c].notna().sum()>0 and frames[0][c].nunique(dropna=True)>1]
    if not good: raise ValueError('Aucune variable exploitable')
    nums=[c for c in good if pd.api.types.is_numeric_dtype(frames[0][c])]
    cats=[c for c in good if c not in nums]
    parts=[]
    if nums: parts.append(('numeric',SimpleImputer(strategy='median'),nums))
    if cats: parts.append(('categorical',OrdinalEncoder(handle_unknown='use_encoded_value',
                            unknown_value=-1,encoded_missing_value=-1,dtype=np.float32),cats))
    prep=ColumnTransformer(parts,remainder='drop',sparse_threshold=0)
    clean=[]
    for f in frames:
        f=f[good].copy()
        for c in nums: f[c]=pd.to_numeric(f[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        for c in cats: f[c]=f[c].astype('string').fillna('__MISSING__').astype(str)
        clean.append(f)
    out=[]
    for j,f in enumerate(clean):
        x=prep.fit_transform(f) if j==0 else prep.transform(f)
        x=np.asarray(x,dtype=np.float32)
        x=np.clip(np.nan_to_num(x,nan=0,posinf=1e4,neginf=-1e4),-1e4,1e4)
        if not np.isfinite(x).all(): raise ValueError('Valeurs X non finies')
        out.append(np.ascontiguousarray(x))
    return out,{'n_features':len(good),'numeric':len(nums),'categorical':len(cats),
                'feature_name_sha256':hashlib.sha256(json.dumps([str(c) for c in good]).encode()).hexdigest()}

def get_bundle(tid):
    task=TASKS[tid]; ds=openml_retry(task.get_dataset)
    X,y,_,_=openml_retry(ds.get_data,target=task.target_name)
    if not isinstance(X,pd.DataFrame) or isinstance(y,pd.DataFrame):
        raise TypeError('Format X/y OpenML inattendu')
    outer,sealed=task.get_train_test_split_indices(fold=0,repeat=0,sample=0)
    outer=np.asarray(outer); sealed=np.asarray(sealed)
    assert np.intersect1d(outer,sealed).size==0 and len(outer)+len(sealed)==len(X)
    label=LabelEncoder().fit(pd.Series(y).iloc[outer].astype(str))
    if len(label.classes_)!=2: raise ValueError(f'{tid}: tâche non binaire')
    y_outer=label.transform(pd.Series(y).iloc[outer].astype(str))
    if min(np.bincount(y_outer))<20: raise ValueError(f'{tid}: classe minoritaire trop rare')
    i_fit,i_report=train_test_split(outer,test_size=0.20,stratify=y_outer,random_state=SEED)
    y_fit=label.transform(pd.Series(y).iloc[i_fit].astype(str))
    i_train,i_val=train_test_split(i_fit,test_size=0.30,stratify=y_fit,random_state=SEED+1)
    y_val_tmp=label.transform(pd.Series(y).iloc[i_val].astype(str))
    i_score,i_select=train_test_split(i_val,test_size=0.50,stratify=y_val_tmp,random_state=SEED+2)
    for a,b in [(i_train,i_score),(i_train,i_select),(i_train,i_report),
                (i_score,i_select),(i_score,i_report),(i_select,i_report)]:
        assert np.intersect1d(a,b).size==0, f'{tid}: fuite entre splits'
    enc=lambda idx: label.transform(pd.Series(y).iloc[idx].astype(str)).astype(np.int64)
    ytr,ysc,ysl,yrp=enc(i_train),enc(i_score),enc(i_select),enc(i_report)
    for nm,z in [('train',ytr),('Vscore',ysc),('Vselect',ysl),('report',yrp)]:
        if min(np.bincount(z))<4: raise ValueError(f'{tid}: classe trop rare dans {nm}')
    (Xtr,Xsc,Xsl,Xrp),feat=encode_task([X.iloc[i_train],X.iloc[i_score],X.iloc[i_select],X.iloc[i_report]])
    if len(X)>MAX_DATASET_ROWS or Xtr.shape[1]>MAX_FEATURES:
        raise ValueError(f'{tid}: hors plafonds ({len(X)} lignes, {Xtr.shape[1]} variables)')
    ctx=choose_indices(ytr,min(LOCAL_TRAIN_CAP,len(ytr)),SEED+tid)
    b=dict(Xtrain=Xtr,ytrain=ytr,Xscore=Xsc,yscore=ysc,Xselect=Xsl,yselect=ysl,
           Xreport=Xrp,yreport=yrp,Xcontext=Xtr[ctx],ycontext=ytr[ctx],
           context_idx=ctx,cat_indices=[],
           meta={**META[tid],'n_train':len(ytr),'n_score':len(ysc),'n_select':len(ysl),
                 'n_report':len(yrp),'n_context':len(ctx),'features':feat,
                 'sealed_outer_test_rows':len(sealed),
                 'split_sha256':hashlib.sha256(np.sort(i_train).astype('<i8').tobytes()).hexdigest()})
    return b

def bundle_cache_paths(tid):
    # Un run precedent, ou une sortie attachee en entree, evite de redependre d'OpenML.
    cands=[ROOT/'bundles'/f'{tid}.npz']
    inp=Path('/kaggle/input')
    if inp.exists(): cands+=sorted(inp.glob(f'**/bundles/{tid}.npz'))
    return cands

def load_bundle(tid):
    for p in bundle_cache_paths(tid):
        if not p.exists(): continue
        try:
            with np.load(p,allow_pickle=False) as a:
                b={k:a[k] for k in a.files if k!='meta_json'}
                b['meta']=json.loads(str(a['meta_json'].item()))
            b['cat_indices']=[]
            return b
        except Exception as exc:
            print(f'  cache illisible {p} ({type(exc).__name__})',flush=True)
    return None

def save_bundle(tid,b):
    d=ROOT/'bundles'; d.mkdir(parents=True,exist_ok=True)
    p=d/f'{tid}.npz'; tmp=p.with_name(p.name+'.tmp')
    arrays={k:v for k,v in b.items() if isinstance(v,np.ndarray)}
    with tmp.open('wb') as f:
        np.savez_compressed(f,meta_json=np.array(json.dumps(b['meta'],default=str)),**arrays)
    tmp.replace(p)

BUNDLES={}; rows=[]; from_cache=0
for tid in ALL_TASK_IDS:
    try:
        cached=load_bundle(tid)
        if cached is not None:
            BUNDLES[tid]=cached; from_cache+=1
        else:
            check_disk(f'chargement {tid}')
            BUNDLES[tid]=get_bundle(tid); save_bundle(tid,BUNDLES[tid])
        m=BUNDLES[tid]['meta']
        rows.append({k:m[k] for k in
            ['task_id','name','n_train','n_score','n_select','n_report','n_context']}
            |{'n_features':m['features']['n_features']})
        print(f"{m['name'][:42]:<42} | train {m['n_train']:>5} Vscore {m['n_score']:>4} "
              f"Vselect {m['n_select']:>4} rapport {m['n_report']:>5} | contexte {m['n_context']:>4} "
              f"| variables {m['features']['n_features']:>3}")
    except Exception as exc:
        record_failure('bundle',tid,META[tid]['name'],exc)
        clean_gpu()

print(f'{from_cache}/{len(ALL_TASK_IDS)} datasets relus depuis un cache local ou attache.')
ACTIVE_IDS=[t for t in ALL_TASK_IDS if t in BUNDLES]
if len(ACTIVE_IDS)<MIN_TARGETS:
    raise RuntimeError(f'Seulement {len(ACTIVE_IDS)} datasets chargés : voir failures.csv.')
PARTITIONS=pd.DataFrame(rows); write_csv(ROOT/'partitions.csv',PARTITIONS); display(PARTITIONS)
write_json(ROOT/'bundles_meta.json',{str(t):BUNDLES[t]['meta'] for t in ACTIVE_IDS})
print(f'\n{len(ACTIVE_IDS)}/24 datasets actifs — chacun sert de cible et de donneur.')

## 4. Moteur épisodiqueRepris du pilote, **inchangé sauf un point** : la taille de l'épisode est devenue un paramètre, puisque c'est l'axe de la phase C. Pas de LoRA, pas de projection, pas de sous-échantillonnage des paramètres.Le repli numérique est celui qui a fait échouer ton run précédent sur `NATICUSdroid` : la T4 est en `sm_75`, elle n'a pas de bf16, l'autocast tombe donc en fp16 et le backward déborde sur les tables larges. On réduit l'échelle de perte sur 8 paliers, puis on repasse en fp32, puis on redessine l'épisode. **Le drapeau `used_amp` compte** : en fp32 les gradients sont déjà à l'échelle 1, l'appelant doit faire `optimizer.step()` et surtout pas `scaler.step()`, qui les diviserait par 65536.

In [ ]:

def stratified_pick(y,n,seed):
    y=np.asarray(y); assert len(np.unique(y))==2
    n=min(int(n),len(y))
    if n==len(y): return np.arange(len(y))
    idx,_=train_test_split(np.arange(len(y)),train_size=n,stratify=y,random_state=int(seed))
    if len(np.unique(y[idx]))<2: raise ValueError('Épisode mono-classe')
    return np.sort(idx)

def ft_episode(b,seed,n_context,n_query):
    # Contexte et requêtes disjoints au niveau des lignes, tous deux dans le train.
    for k in range(30):
        s=int(seed)+k*100003
        iq=stratified_pick(b['ytrain'],n_query,s)
        pool=np.flatnonzero(~np.isin(np.arange(len(b['ytrain'])),iq))
        if len(pool)<n_context or len(np.unique(b['ytrain'][pool]))<2: continue
        ic=pool[stratified_pick(b['ytrain'][pool],n_context,s+1)]
        assert not (set(ic.tolist()) & set(iq.tolist()))
        return (b['Xtrain'][ic],b['ytrain'][ic],b['Xtrain'][iq],b['ytrain'][iq],s),ic,iq
    raise RuntimeError('Impossible de construire un épisode disjoint')

class EpisodeEngine:
    def __init__(self,checkpoint,cat_indices,seed):
        set_seed(seed)
        kw=dict(device=DEVICE,n_estimators=1,random_state=int(seed),fit_mode='batched',
                differentiable_input=False,memory_saving_mode=True,
                categorical_features_indices=cat_indices,
                inference_config={'ENABLE_GPU_PREPROCESSING':False,'TRANSFORM_TEXT':False,
                                  'TRANSFORM_DATES':False})
        self.est=(TabPFNClassifier.create_default_for_version(ModelVersion.V3_5,**kw)
                  if checkpoint is None else TabPFNClassifier(model_path=str(checkpoint),**kw))
        self.est._initialize_model_variables(); self.est.model_.to(DEVICE)
        for p in _parameters_unused_by_task(self.est.model_,'multiclass'): p.requires_grad=False
        self.wrapper=FinetunedTabPFNClassifier(device=DEVICE,model_version=ModelVersion.V3_5)
        self.wrapper.finetuned_estimator_=self.est
        self.wrapper._ddp_module_=None; self.wrapper._local_n_estimators_=1
        self.performance=PerformanceOptions(force_recompute_layer=True,use_chunkwise_inference=False)
    def params(self):
        return {n:p for n,p in self.est.model_.named_parameters() if p.requires_grad}
    def batch(self,episode):
        xc,yc,xq,yq,seed=episode; k=len(yc)
        assert set(np.unique(yc))=={0,1}
        def fixed_split(X,y,stratify=None): return X[:k],X[k:],y[:k],y[k:]
        ds=get_preprocessed_dataset_chunks(calling_instance=self.est,
            X_raw=np.ascontiguousarray(np.concatenate([xc,xq]),dtype=np.float32),
            y_raw=np.concatenate([yc,yq]),split_fn=fixed_split,max_data_size=None,
            model_type='classifier',equal_split_size=False,data_shuffle_seed=seed,
            preprocessing_random_state=seed,shuffle=False)
        assert len(ds)==1
        return meta_dataset_collator([ds[0]])
    def loss(self,batch):
        self.est.fit_from_preprocessed(batch.X_context,batch.y_context,batch.cat_indices,
                                       batch.configs,performance_options=self.performance)
        self.est.model_.train()
        return self.wrapper._forward_with_loss(batch)
    def save(self,path):
        p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
        tmp=p.with_name(p.name+'.tmp'); save_tabpfn_model(self.est,tmp); tmp.replace(p)
    def close(self):
        del self.wrapper; del self.est; clean_gpu()

AMP_EVENTS=[]   # journal des replis numériques : une donnée de l'article

def _grads_finite(engine):
    return all(p.grad is None or bool(torch.isfinite(p.grad).all())
               for p in engine.params().values())

def backward_checked(engine,batch,optimizer,scaler,tag='',step=None):
    # Renvoie (loss, used_amp). used_amp=False => gradients déjà à l'échelle 1 :
    # l'appelant doit faire optimizer.step(), surtout pas scaler.step().
    scale0=scaler.get_scale()
    for k in range(8):                       # 65536 -> 4, par quarts successifs
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=True),sdpa_kernel_context(): loss=engine.loss(batch)
        if not torch.isfinite(loss): break   # le problème est en avant, pas en arrière
        with sdpa_kernel_context(): scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if _grads_finite(engine):
            if k: AMP_EVENTS.append(dict(tag=tag,step=step,mode='amp_rescaled',
                                         reductions=k,scale=scaler.get_scale()))
            return float(loss.detach()),True
        scaler.update(new_scale=max(1.0,scaler.get_scale()/4))
    # Repli fp32 : l'overflow fp16 disparaît, le pas reste exact.
    optimizer.zero_grad(set_to_none=True)
    with autocast(enabled=False),sdpa_kernel_context(): loss=engine.loss(batch)
    if torch.isfinite(loss):
        with sdpa_kernel_context(): loss.backward()
        if _grads_finite(engine):
            scaler.update(new_scale=scale0)
            AMP_EVENTS.append(dict(tag=tag,step=step,mode='fp32'))
            return float(loss.detach()),False
    raise FloatingPointError('Gradients non finis en AMP comme en fp32')

def named_hash(engine):
    h=hashlib.sha256()
    for name,p in sorted(engine.params().items()):
        a=p.detach().cpu().contiguous()
        h.update(name.encode()); h.update(str(tuple(a.shape)).encode())
        h.update(a.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

print('Moteur épisodique prêt.')

## 5. Entraînement jalonné, et la courbe de contexte`train_fixed` exécute un nombre de pas **fixé d'avance**, sans arrêt anticipé et sans sélection de jalon sur une métrique. Il enregistre les poids à chaque jalon : c'est ce qui rend la grille abordable, puisque les trois budgets ne coûtent que le plus long des trois.`curve_for` est le cœur du dispositif : il prend **un** jeu de poids et le mesure à **toutes** les tailles de contexte, sur les mêmes requêtes du rapport. Le modèle de base et les modèles adaptés passent exactement par le même chemin de code — ils sont tous chargés depuis un fichier de poids — pour qu'aucun écart ne puisse venir de là.Deux économies visibles dans le code :- une taille de contexte **supérieure ou égale** au train de la cible est sautée : elle donnerait le même contexte que `full`, donc le même nombre, pour le même prix ;- les contextes petits (<= 256) sont **retirés trois fois** avec des tirages différents et les métriques sont moyennées. À 32 lignes, la variance du tirage domine tout le reste ; sans répétition, on mesurerait surtout du bruit.Les jalons sont effacés dès que le dataset est évalué : trois jeux de poids TabPFN pèsent près de 3 Gio, et `/kaggle/temp` n'est pas extensible.

In [ ]:

BASE=SCRATCH/'base_weights.pth'
if not BASE.exists():
    check_disk('poids de base',extra=2.0)
    e=EpisodeEngine(None,[],SEED)
    try: e.save(BASE)
    finally: e.close()
print('Poids officiels :',BASE,'|',round(BASE.stat().st_size/GIB,2),'Gio')

def predict_ctx(checkpoint,Xctx,yctx,Xq,seed,batch,cat_indices):
    m=TabPFNClassifier(model_path=str(checkpoint),device=DEVICE,n_estimators=1,
        fit_mode='low_memory',memory_saving_mode=True,random_state=int(seed),
        categorical_features_indices=cat_indices,ignore_pretraining_limits=True)
    try:
        m.fit(Xctx,yctx); pos=list(m.classes_).index(1); parts=[]
        for st in range(0,len(Xq),batch):
            parts.append(np.asarray(m.predict_proba(Xq[st:st+batch])[:,pos],dtype=float))
        p=np.concatenate(parts)
        assert len(p)==len(Xq) and np.isfinite(p).all()
        return p
    finally:
        del m; clean_gpu()

def ctx_indices(b,size,seed):
    # size=None => tout le train. Sinon sous-échantillon stratifié déterministe.
    n=len(b['ytrain']); k=n if size is None else min(int(size),n)
    return choose_indices(b['ytrain'],k,seed)

def eval_slice(b,cap,seed):
    idx=choose_indices(b['yreport'],min(cap,len(b['yreport'])),seed)
    return b['Xreport'][idx],b['yreport'][idx]

def curve_for(checkpoint,tid,arm,ft_steps,episode_rows):
    # Un jeu de poids -> une courbe complète sur l'axe du contexte.
    b=BUNDLES[tid]; cat=b['cat_indices']; ntr=len(b['ytrain'])
    Xq,yq=eval_slice(b,REPORT_EVAL_CAP,SEED+900+tid)
    out=[]
    for size in CONTEXT_SIZES:
        if size is not None and size>=ntr:
            continue                      # identique au contexte complet : on ne paie pas deux fois
        nrep=REPEATS_SMALL if (size is not None and size<=SMALL_CONTEXT_MAX) else 1
        for r in range(nrep):
            try:
                idx=ctx_indices(b,size,SEED+tid+7919*r)
            except ValueError as exc:
                print(f'  {arm} ctx={size} : tirage impossible ({exc})',flush=True); break
            t0=time.monotonic(); p=None
            for bs in (PRED_BATCH,16,4,1):
                try:
                    p=predict_ctx(checkpoint,b['Xtrain'][idx],b['ytrain'][idx],Xq,SEED,bs,cat)
                    break
                except RuntimeError as exc:
                    if not is_oom(exc): raise
                    clean_gpu()
            if p is None:
                print(f'  {arm} ctx={size} : OOM à {len(idx)} lignes, point omis',flush=True); break
            out.append(dict(run_hash=RUN_HASH,target=tid,name=META[tid]['name'],n_train=ntr,
                            arm=arm,ft_steps=int(ft_steps),episode_rows=int(episode_rows),
                            context_size=('full' if size is None else str(int(size))),
                            requested=(ntr if size is None else int(size)),
                            n_context_rows=len(idx),repeat=r,n_report_eval=len(yq),
                            seconds=round(time.monotonic()-t0,1),**scores(yq,p)))
    if not out: raise RuntimeError(f'{arm} : aucune taille de contexte évaluable')
    return out

def train_fixed(b,start,path,milestones,seed,tag,n_context,n_query):
    path=Path(path); path.mkdir(parents=True,exist_ok=True)
    done=path/'complete.json'; end=max(milestones)
    signature=sha(dict(run=RUN_HASH,start=sha_file(start),steps=sorted(milestones),
                       seed=seed,tag=tag,ep=(n_context,n_query)))
    if done.exists():
        info=json.loads(done.read_text())
        if info['signature']==signature and all(
                Path(c['path']).is_file() and sha_file(c['path'])==c['sha256']
                for c in info['checkpoints'].values()):
            print(tag,': déjà entraîné, réutilisé.'); return info
    check_disk(tag,extra=2.0*len(milestones))
    e=EpisodeEngine(start,b['cat_indices'],seed)
    opt=torch.optim.AdamW(list(e.params().values()),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scaler=GradScaler(); history=[]; ckpts={}; seen=set(); redraws=0
    t0=time.monotonic(); torch.cuda.reset_peak_memory_stats(); initial=named_hash(e)
    try:
        for step in range(1,end+1):
            # Un épisode non fini même en fp32 est redessiné, jamais sauté :
            # le budget de pas reste exactement celui annoncé.
            for attempt in range(6):
                episode,ic,iq=ft_episode(b,seed+step*7919+attempt*104729,n_context,n_query)
                batch=e.batch(episode)
                try:
                    loss,used_amp=backward_checked(e,batch,opt,scaler,tag=tag,step=step)
                    break
                except FloatingPointError:
                    redraws+=1
                    AMP_EVENTS.append(dict(tag=tag,step=step,mode='episode_redrawn',attempt=attempt))
                    del batch; clean_gpu()
            else:
                raise FloatingPointError(
                    f'{tag} : 6 épisodes consécutifs non finis au pas {step}. '
                    'Instabilité reproductible, pas un aléa : signaler ce dataset.')
            seen.update(ic.tolist()); seen.update(iq.tolist())
            norm=float(torch.nn.utils.clip_grad_norm_(list(e.params().values()),GRAD_CLIP))
            if used_amp:
                scaler.step(opt); scaler.update()
            else:
                opt.step()                      # gradients déjà à l'échelle 1
            history.append(dict(step=step,loss=loss,grad_norm_before_clip=norm,
                                amp=used_amp,seconds=time.monotonic()-t0))
            del batch
            if step in milestones:
                ck=path/f'step_{step:04d}.pth'; e.save(ck)
                wh=named_hash(e)
                assert wh!=initial,'Poids inchangés après entraînement'
                ckpts[str(step)]=dict(path=str(ck),sha256=sha_file(ck),weights_hash=wh,
                                      seconds=time.monotonic()-t0)
                write_csv(path/'training_history.csv',pd.DataFrame(history))
                fp32=int((~pd.DataFrame(history).amp).sum())
                print(f'  {tag} | pas {step}/{end} | loss {loss:.5f} | |g| {norm:.3f}'
                      +(f' | replis fp32 {fp32} | redraws {redraws}' if fp32 or redraws else ''),
                      flush=True)
        info=dict(signature=signature,initial_hash=initial,checkpoints=ckpts,
                  seconds=time.monotonic()-t0,n_train=len(b['ytrain']),
                  unique_rows_seen=len(seen),episode_rows=n_context+n_query,
                  fp32_steps=int((~pd.DataFrame(history).amp).sum()),
                  episode_redraws=redraws,
                  final_loss_mean=float(pd.DataFrame(history).loss.tail(10).mean()),
                  peak_allocated_gib=torch.cuda.max_memory_allocated()/GIB)
        write_json(done,info)
        return info
    finally:
        del opt,scaler; e.close()
        if AMP_EVENTS: write_csv(ROOT/'amp_events.csv',pd.DataFrame(AMP_EVENTS))

print('Entraînement jalonné et courbe de contexte prêts.')

## 6. PrétestsTrois choses sont vérifiées avant de lancer cinq heures de calcul, et rien ne part sans le PASS :1. **mémoire** — quel lot de requêtes tient sur la T4, y compris au contexte complet du plus gros dataset ;2. **fonctionnel** — sur `breast_cancer`, étranger aux 24 tâches : les poids bougent, le rechargement est exact, les prédictions sont reproductibles, et un jalon donne bien une courbe à plusieurs points ;3. **le contrôle qui compte ici** — que deux jeux de poids différents (base et adapté) donnent bien des prédictions **différentes** au même contexte. Sans ça, toute la grille serait plate pour une raison triviale.

In [ ]:

attempts=[]; PRED_BATCH=None
for bs in PRED_BATCHES:
    try:
        for tid in ACTIVE_IDS:
            b=BUNDLES[tid]; predict_ctx(BASE,b['Xcontext'],b['ycontext'],b['Xreport'][:bs],SEED,bs,b['cat_indices'])
        PRED_BATCH=bs; break
    except RuntimeError as exc:
        if not is_oom(exc): raise
        attempts.append(dict(batch=bs,error='CUDA OOM')); clean_gpu()
if PRED_BATCH is None: raise RuntimeError('Même une requête unique ne tient pas en mémoire.')
print('Lot de requêtes retenu :',PRED_BATCH)

big=max(ACTIVE_IDS,key=lambda t:len(BUNDLES[t]['ytrain']))
nbig=len(BUNDLES[big]['ytrain'])
try:
    predict_ctx(BASE,BUNDLES[big]['Xtrain'],BUNDLES[big]['ytrain'],
                BUNDLES[big]['Xreport'][:PRED_BATCH],SEED,PRED_BATCH,BUNDLES[big]['cat_indices'])
    print(f'Contexte complet testé sur {META[big]["name"]} : {nbig} lignes, OK')
except RuntimeError as exc:
    if not is_oom(exc): raise
    clean_gpu()
    print(f'ATTENTION : contexte complet de {nbig} lignes en OOM. Les points ctx=full '
          'se replieront sur des lots plus petits, ou seront omis.',flush=True)

from sklearn.datasets import load_breast_cancer
xp,yp=load_breast_cancer(return_X_y=True)
a_,v_=train_test_split(np.arange(len(yp)),train_size=400,stratify=yp,random_state=SEED)
probe=dict(Xtrain=xp[a_].astype('float32'),ytrain=yp[a_].astype(np.int64),
           Xreport=xp[v_].astype('float32'),yreport=yp[v_].astype(np.int64),
           Xcontext=xp[a_].astype('float32'),ycontext=yp[a_].astype(np.int64),cat_indices=[])
RUN_HASH='preflight'
pf=SCRATCH/'_preflight'
if pf.exists(): shutil.rmtree(pf)
try:
    info=train_fixed(probe,BASE,pf,[1,2],SEED,'prétest',FT_CONTEXT,FT_QUERY)
    ck=info['checkpoints']['2']['path']
    e=EpisodeEngine(ck,[],SEED)
    try: assert named_hash(e)==info['checkpoints']['2']['weights_hash'],'Rechargement incorrect'
    finally: e.close()
    q=probe['Xreport'][:16]
    p1=predict_ctx(ck,probe['Xtrain'][:128],probe['ytrain'][:128],q,SEED,16,[])
    p2=predict_ctx(ck,probe['Xtrain'][:128],probe['ytrain'][:128],q,SEED,16,[])
    assert np.allclose(p1,p2,atol=1e-5,rtol=1e-4),'Prédictions non reproductibles'
    p0=predict_ctx(BASE,probe['Xtrain'][:128],probe['ytrain'][:128],q,SEED,16,[])
    dmax=float(np.abs(p0-p1).max())
    assert dmax>1e-4,('Le modèle adapté prédit exactement comme le modèle de base : '
                      'la grille serait plate pour une raison triviale.')
    print(f'PRÉTEST PASS — poids rechargés à l identique, prédictions déterministes, '
          f'écart base/adapté max {dmax:.4g}')
finally:
    shutil.rmtree(pf,ignore_errors=True); clean_gpu()

TASK_IDS=[t for t in ACTIVE_IDS if t in SHARD_IDS]
LONG_IDS=(sorted(TASK_IDS,key=lambda t:-len(BUNDLES[t]['ytrain']))[:LONG_PHASE_TASKS]
          if RUN_LONG_EPISODE_PHASE else [])
CONFIG=dict(protocol='context_x_finetuning_v1',seed=SEED,tasks=TASK_IDS,
    context_sizes=[str(c) for c in CONTEXT_SIZES],ref_context=REF_CONTEXT,
    repeats_small=REPEATS_SMALL,small_max=SMALL_CONTEXT_MAX,
    ft_milestones=list(FT_MILESTONES),ft_episode=(FT_CONTEXT,FT_QUERY),
    long_phase=bool(RUN_LONG_EPISODE_PHASE),long_ids=LONG_IDS,
    long_episode=(LONG_FT_CONTEXT,LONG_FT_QUERY),long_milestones=list(LONG_FT_MILESTONES),
    lr=LEARNING_RATE,wd=WEIGHT_DECAY,clip=GRAD_CLIP,
    report_cap=REPORT_EVAL_CAP,pred_batch=PRED_BATCH,
    splits='outer_80_20_then_70_15_15_v1',
    versions={k:importlib.metadata.version(k) for k in
              ['tabpfn','torch','numpy','pandas','scikit-learn','scipy']})
RUN_HASH=sha(CONFIG)[:16]
manifest=ROOT/'manifest.json'
if manifest.exists():
    assert json.loads(manifest.read_text())['run_hash']==RUN_HASH,'Configuration différente : changer RUN_TAG.'
else:
    write_json(manifest,dict(run_hash=RUN_HASH,config=CONFIG,memory_attempts=attempts))
RUN_T0=time.monotonic()
def hours_left():
    return TIME_BUDGET_HOURS-(time.monotonic()-RUN_T0)/3600
print('Protocole figé :',RUN_HASH,'|',len(TASK_IDS),'datasets | phase C sur',len(LONG_IDS))

## 7. Prédiction enregistréeÉcrite **avant** de regarder quoi que ce soit, et imprimée dans la sortie du notebook pour qu'elle soit horodatée par le run lui-même. C'est ce qui distingue une prédiction d'une explication rétrospective.

In [ ]:

PREREG = '''
H1  Le gain du fine-tuning décroît quand le contexte grandit.
    Test : Spearman(log2 contexte, gain de log-loss) par dataset, au budget de 200 pas,
    puis test des signes sur les 24 coefficients. Prédiction : majorité de coefficients
    NÉGATIFS, test des signes significatif.

H2  Il existe un contexte de croisement.
    Test : le gain moyen est > 0 aux petits contextes (32-128) et <= 0 au contexte de
    référence (1024). Prédiction : croisement entre 128 et 512.

H3  Le gain à 1024 lignes reproduit tes trois runs précédents.
    Prédiction : gain moyen proche de zéro, légèrement négatif (-0,006 de médiane AUC
    dans ton étude locale). Si ce point ne se reproduit PAS, le harnais a changé de
    comportement et rien d autre n est interprétable.

H4  (phase C) Le pic de gain se déplace avec la taille de l épisode d entraînement.
    Prédiction : avec des épisodes de 640 lignes, le croisement se déplace vers la
    DROITE. Si le croisement ne bouge pas, l effet tient au contexte d inférence seul.

ISSUE ALTERNATIVE, à écrire telle quelle si elle se produit :
    gain <= 0 à TOUTES les tailles de contexte, y compris 32 lignes. Le négatif devient
    alors plus fort qu avant, puisqu il survit au régime le plus favorable au fine-tuning.
    Ce n est pas un échec de l expérience, c est son résultat.
'''
print(PREREG)
write_json(ROOT/'preregistration.json',dict(run_hash=RUN_HASH,text=PREREG,
    written_before_any_result=True))

## 8. Phase A — la courbe de contexte du modèle non adaptéAucun gradient ici. C'est la référence de tous les gains, et c'est aussi, à elle seule, la première moitié de l'article : la courbe complète de 32 lignes à tout le train, sur 24 datasets, que ton run précédent n'avait qu'à partir de 256.Elle est faite en premier parce qu'elle est peu coûteuse et qu'elle a de la valeur même si la suite du run échoue.

In [ ]:

CURVE_PATH=ROOT/'curves.csv'
curves=pd.read_csv(CURVE_PATH).to_dict('records') if CURVE_PATH.exists() else []
def arm_done(tid,arm):
    return any(int(r['target'])==tid and r['arm']==arm for r in curves)

print('=== PHASE A — modèle non adapté ===',flush=True)
for k,tid in enumerate(TASK_IDS,1):
    if arm_done(tid,'base'):
        print(f'{k}/{len(TASK_IDS)} {META[tid]["name"]} : déjà fait.'); continue
    if hours_left()<=0:
        print('Garde de temps atteinte : phase A interrompue proprement.',flush=True); break
    try:
        t0=time.monotonic()
        new=curve_for(BASE,tid,'base',0,0)
        curves.extend(new); write_csv(CURVE_PATH,pd.DataFrame(curves))
        c={r['context_size']:round(r['logloss'],4) for r in new if r['repeat']==0}
        print(f'{k}/{len(TASK_IDS)} {META[tid]["name"][:34]:<34} train {len(BUNDLES[tid]["ytrain"]):>5} '
              f'| {c} | {time.monotonic()-t0:.0f}s',flush=True)
    except Exception as exc:
        record_failure('base_curve',tid,META[tid]['name'],exc); clean_gpu()
print(f'\nPhase A : {len({r["target"] for r in curves if r["arm"]=="base"})} datasets, '
      f'{hours_left():.2f} h de budget restantes.')

## 9. Phase B — la grille complèteUn dataset à la fois : on entraîne une fois jusqu'au plus grand budget en enregistrant les jalons, on évalue chaque jalon sur **toute** la gamme de contextes, puis on efface les poids. Un seul jeu de jalons existe à la fois sur le disque.C'est la phase longue. Elle est reprenable : les lignes déjà écrites dans `curves.csv` ne sont jamais recalculées, et la garde de temps l'arrête entre deux datasets plutôt qu'au milieu d'un entraînement.

In [ ]:

FT_DIR=SCRATCH/'ft'; FT_DIR.mkdir(parents=True,exist_ok=True)
TRAIN_PATH=ROOT/'training_summary.csv'
train_rows=pd.read_csv(TRAIN_PATH).to_dict('records') if TRAIN_PATH.exists() else []

def run_ft_phase(ids,milestones,n_context,n_query,prefix,label):
    global curves,train_rows
    print(f'\n=== {label} ===',flush=True)
    for k,tid in enumerate(ids,1):
        arms=[f'{prefix}_{s}' for s in milestones]
        if all(arm_done(tid,a) for a in arms):
            print(f'{k}/{len(ids)} {META[tid]["name"]} : déjà fait.'); continue
        if hours_left()<=0:
            print('Garde de temps atteinte : phase interrompue proprement.',flush=True); break
        b=BUNDLES[tid]; path=FT_DIR/f'{prefix}_{tid}'
        print(f'\n----- {k}/{len(ids)} {META[tid]["name"]} | train {len(b["ytrain"])} '
              f'| épisode {n_context+n_query} lignes | {hours_left():.2f} h restantes -----',flush=True)
        try:
            check_disk(f'fine-tuning {tid}',extra=2.0*len(milestones))
            info=train_fixed(b,BASE,path,milestones,SEED+tid,f'{prefix} {META[tid]["name"][:24]}',
                             n_context,n_query)
            train_rows=[r for r in train_rows if not (int(r['target'])==tid and r['prefix']==prefix)]
            train_rows.append(dict(target=tid,name=META[tid]['name'],prefix=prefix,
                episode_rows=n_context+n_query,steps=max(milestones),
                seconds=info['seconds'],fp32_steps=info['fp32_steps'],
                episode_redraws=info['episode_redraws'],unique_rows_seen=info['unique_rows_seen'],
                final_loss_mean=info['final_loss_mean'],peak_gib=info['peak_allocated_gib']))
            write_csv(TRAIN_PATH,pd.DataFrame(train_rows))
            for s in milestones:
                arm=f'{prefix}_{s}'
                if arm_done(tid,arm): continue
                ck=info['checkpoints'][str(s)]['path']
                new=curve_for(ck,tid,arm,s,n_context+n_query)
                curves.extend(new); write_csv(CURVE_PATH,pd.DataFrame(curves))
                base={ (r['context_size']):r['logloss'] for r in curves
                       if int(r['target'])==tid and r['arm']=='base' and r['repeat']==0 }
                g={r['context_size']:round(base.get(r['context_size'],np.nan)-r['logloss'],4)
                   for r in new if r['repeat']==0}
                print(f'  {arm} | gain de log-loss par contexte : {g}',flush=True)
        except Exception as exc:
            record_failure(prefix,tid,META[tid]['name'],exc); clean_gpu()
        finally:
            shutil.rmtree(path,ignore_errors=True); clean_gpu()

run_ft_phase(TASK_IDS,FT_MILESTONES,FT_CONTEXT,FT_QUERY,'ft',
             f'PHASE B — fine-tuning, épisodes de {FT_CONTEXT+FT_QUERY} lignes')
CUR=pd.DataFrame(curves); write_csv(CURVE_PATH,CUR)
print(f'\nPhase B terminée. {CUR.target.nunique()} datasets, {len(CUR)} points, '
      f'{len(FAILURES)} écart(s). Budget restant : {hours_left():.2f} h.')

## 10. Phase C — la taille de l'épisode d'entraînementC'est la réponse à l'objection de relecture. Mêmes datasets parmi les plus grands, même protocole, une seule chose change : les épisodes d'entraînement font 640 lignes au lieu de 128.Si le gain du fine-tuning est une affaire de **contexte d'inférence**, la courbe doit garder la même forme. S'il est une affaire de **spécialisation à la taille d'épisode**, le pic doit se déplacer vers la droite. Les deux réponses sont informatives ; la seconde est même la plus intéressante, parce qu'elle donnerait une recette : *accorder la taille des épisodes de fine-tuning à celle du contexte de déploiement.*Cette phase passe en dernier et se saute d'elle-même si le budget de temps est épuisé.

In [ ]:

if RUN_LONG_EPISODE_PHASE and LONG_IDS and hours_left()>0.5:
    run_ft_phase(LONG_IDS,LONG_FT_MILESTONES,LONG_FT_CONTEXT,LONG_FT_QUERY,'ftlong',
                 f'PHASE C — épisodes de {LONG_FT_CONTEXT+LONG_FT_QUERY} lignes')
else:
    print('Phase C non exécutée (désactivée, ou budget de temps insuffisant).')
CUR=pd.DataFrame(curves); write_csv(CURVE_PATH,CUR)
print(f'\n{CUR.target.nunique()} datasets, {CUR.arm.nunique()} bras, {len(CUR)} points mesurés.')

## 11. RésultatsQuatre lectures, dans l'ordre où elles doivent être faites.1. **La courbe de base** — le contexte, seul. C'est le résultat déjà connu, étendu vers le bas.2. **La grille** — gain du fine-tuning par (budget, taille de contexte). C'est la table principale.3. **La pente** — Spearman par dataset entre contexte et gain, puis test des signes. C'est H1.4. **L'équivalence** — combien de doublements de contexte vaut le fine-tuning. C'est le chiffre qu'on retient d'un article.

In [ ]:

from scipy.stats import wilcoxon, spearmanr, binomtest

CUR=pd.read_csv(CURVE_PATH)
KEY=['target','name','n_train','arm','ft_steps','episode_rows','context_size','requested']
AGG=(CUR.groupby(KEY,as_index=False)
        .agg(logloss=('logloss','mean'),auc=('auc','mean'),ap=('ap','mean'),
             n_context_rows=('n_context_rows','mean'),repeats=('repeat','count')))
AGG['ctx']=AGG.n_context_rows.astype(float)
write_csv(ROOT/'curves_agg.csv',AGG)

print('=== 1. LA COURBE DE BASE — le contexte seul, modèle non adapté ===')
B=AGG[AGG.arm=='base'].copy()
ref=(B[B.context_size==str(REF_CONTEXT)].set_index('target')['logloss'])
B['gain_vs_ref']=B.target.map(ref)-B.logloss
base_tab=(B.groupby('context_size')
           .agg(lignes_moy=('ctx','mean'),logloss_moy=('logloss','mean'),auc_moy=('auc','mean'),
                gain_vs_1024=('gain_vs_ref','mean'),datasets=('target','nunique'))
           .sort_values('lignes_moy').round(5))
display(base_tab); write_csv(ROOT/'table_base_curve.csv',base_tab.reset_index())

print('\n=== 2. LA GRILLE — gain du fine-tuning, par budget et par contexte ===')
FT=AGG[AGG.arm.str.startswith('ft')].merge(
    AGG[AGG.arm=='base'][['target','context_size','logloss','auc']],
    on=['target','context_size'],suffixes=('','_base'),how='inner')
FT['gain_logloss']=FT.logloss_base-FT.logloss
FT['gain_auc']=FT.auc-FT.auc_base
FT['phase']=np.where(FT.arm.str.startswith('ftlong'),'C (épisodes longs)','B (épisodes courts)')
write_csv(ROOT/'ft_gains.csv',FT)

def grid(sub,titre):
    if sub.empty: print(f'\n{titre} : aucune donnée.'); return None
    t=(sub.groupby(['ft_steps','context_size'])
          .agg(lignes_moy=('ctx','mean'),gain_moyen=('gain_logloss','mean'),
               gain_median=('gain_logloss','median'),gain_auc_moyen=('gain_auc','mean'),
               datasets_ameliores=('gain_logloss',lambda s:int((s>0).sum())),
               datasets=('target','nunique'))
          .reset_index().sort_values(['ft_steps','lignes_moy']).round(5))
    print(f'\n{titre} — positif = le fine-tuning aide')
    display(t); return t

gB=grid(FT[FT.phase.str.startswith('B')],f'PHASE B — épisodes de {FT_CONTEXT+FT_QUERY} lignes')
gC=grid(FT[FT.phase.str.startswith('C')],f'PHASE C — épisodes de {LONG_FT_CONTEXT+LONG_FT_QUERY} lignes')
if gB is not None: write_csv(ROOT/'grille_phaseB.csv',gB)
if gC is not None: write_csv(ROOT/'grille_phaseC.csv',gC)

print('\nTest du gain contre zéro, à chaque taille de contexte (phase B, budget maximal)')
top=max(FT_MILESTONES); zero=[]
for cs,g in FT[(FT.phase.str.startswith('B'))&(FT.ft_steps==top)].groupby('context_size'):
    d=g.gain_logloss.dropna()
    if len(d)>=5 and d.abs().sum()>0:
        zero.append(dict(context_size=cs,lignes_moy=round(float(g.ctx.mean()),1),
            gain_moyen=round(float(d.mean()),5),gain_median=round(float(d.median()),5),
            wilcoxon_p=round(float(wilcoxon(d).pvalue),6),
            positifs=int((d>0).sum()),n=len(d)))
ZERO=pd.DataFrame(zero).sort_values('lignes_moy') if zero else pd.DataFrame()
if len(ZERO): display(ZERO); write_csv(ROOT/'test_vs_zero.csv',ZERO)

print('\n=== 3. H1 — LA PENTE : le gain décroît-il avec le contexte ? ===')
slopes=[]
for tid,g in FT[(FT.phase.str.startswith('B'))&(FT.ft_steps==top)].groupby('target'):
    g=g.dropna(subset=['gain_logloss'])
    if len(g)>=4 and g.ctx.nunique()>=4:
        rho,p=spearmanr(np.log2(g.ctx),g.gain_logloss)
        slopes.append(dict(target=tid,name=g.name.iloc[0],n_train=int(g.n_train.iloc[0]),
            rho=round(float(rho),4),p=round(float(p),4),points=len(g),
            gain_petit=round(float(g[g.ctx<=SMALL_CONTEXT_MAX].gain_logloss.mean()),5),
            gain_grand=round(float(g[g.ctx>=REF_CONTEXT].gain_logloss.mean()),5)))
SL=pd.DataFrame(slopes)
if len(SL):
    display(SL.sort_values('rho')); write_csv(ROOT/'pentes_par_dataset.csv',SL)
    neg=int((SL.rho<0).sum()); n=len(SL)
    bt=binomtest(neg,n,0.5)
    print(f'\nH1 : {neg}/{n} datasets ont une corrélation NÉGATIVE entre contexte et gain '
          f'(test des signes p={bt.pvalue:.5f}, rho médian {SL.rho.median():+.3f}).')
    print('  -> prédiction confirmée' if neg>n/2 and bt.pvalue<0.05 else
          '  -> prédiction NON confirmée : le dire tel quel.')
    d2=(SL.gain_petit-SL.gain_grand).dropna()
    if len(d2)>=5 and d2.abs().sum()>0:
        print(f'  écart petit contexte - grand contexte : {d2.mean():+.5f} en moyenne, '
              f'{int((d2>0).sum())}/{len(d2)} positifs, Wilcoxon p={wilcoxon(d2).pvalue:.5f}')
else:
    print('Pas assez de points par dataset pour estimer une pente.')

print('\n=== 4. LE CROISEMENT, et combien de contexte vaut le fine-tuning ===')
cross=[]
for (tid,st),g in FT[FT.phase.str.startswith('B')].groupby(['target','ft_steps']):
    g=g.dropna(subset=['gain_logloss']).sort_values('ctx')
    if g.empty: continue
    pos=g[g.gain_logloss>0]; neg_=g[g.gain_logloss<=0]
    bb=B[B.target==tid].sort_values('ctx')
    eq=np.nan
    at_ref=g[g.context_size==str(REF_CONTEXT)]
    if len(bb)>=3 and len(at_ref):
        # combien de lignes de contexte faudrait-il au modèle NON adapté pour égaler
        # le modèle adapté au contexte de référence ?
        x=np.log2(bb.ctx.to_numpy()); y=bb.logloss.to_numpy()
        o=np.argsort(-x); yy=np.maximum.accumulate(y[o]); xx=x[o]   # log-loss croissante
        target_ll=float(at_ref.logloss.iloc[0])
        if yy.min()<=target_ll<=yy.max():
            eq=float(np.interp(target_ll,yy,xx))-np.log2(float(at_ref.ctx.iloc[0]))
    cross.append(dict(target=tid,name=g.name.iloc[0],ft_steps=int(st),
        n_train=int(g.n_train.iloc[0]),
        dernier_contexte_gagnant=(int(pos.ctx.max()) if len(pos) else None),
        premier_contexte_perdant=(int(neg_.ctx.min()) if len(neg_) else None),
        doublements_equivalents=(round(eq,3) if np.isfinite(eq) else None)))
CROSS=pd.DataFrame(cross)
if len(CROSS):
    display(CROSS.sort_values(['ft_steps','n_train'])); write_csv(ROOT/'croisements.csv',CROSS)
    eqs=CROSS[CROSS.ft_steps==top].doublements_equivalents.dropna()
    if len(eqs):
        print(f'\nAu budget de {top} pas, le fine-tuning vaut en médiane '
              f'{eqs.median():+.2f} doublement(s) de contexte '
              f'({int((eqs>0).sum())}/{len(eqs)} datasets où il en vaut un positif).')
        print('  Lecture : un nombre négatif veut dire que le fine-tuning RETIRE '
              'l équivalent de tant de doublements de contexte.')

if len(FT[FT.phase.str.startswith('C')]):
    print('\n=== H4 — la taille de l épisode déplace-t-elle la courbe ? ===')
    cmp_=FT[FT.target.isin(LONG_IDS)&FT.ft_steps.isin(set(FT_MILESTONES)&set(LONG_FT_MILESTONES))]
    t4=(cmp_.groupby(['phase','ft_steps','context_size'])
            .agg(lignes_moy=('ctx','mean'),gain_moyen=('gain_logloss','mean'),
                 datasets=('target','nunique'))
            .reset_index().sort_values(['ft_steps','lignes_moy','phase']).round(5))
    display(t4); write_csv(ROOT/'comparaison_episodes.csv',t4)

## 12. FiguresTrois figures, et une quatrième si la phase C a tourné. Les budgets de fine-tuning sont une quantité **ordonnée** : ils sont donc codés par une rampe d'une seule teinte, du clair au foncé, avec des marqueurs distincts en encodage secondaire. L'orange ne sert qu'à une chose : la référence du modèle non adapté.

In [ ]:

RAMP=['#8FB4DC','#4B7FBB','#1F4E79']      # rampe séquentielle : budget croissant
MARKS=['o','s','^']
ORANGE='#D1793A'; GREY='#6B7280'; BLUE='#3B6FB6'
plt.rcParams.update({'figure.dpi':160,'savefig.dpi':300,'font.size':9,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'axes.edgecolor':'#9CA3AF','axes.labelcolor':'#374151',
                     'xtick.color':GREY,'ytick.color':GREY,'text.color':'#1F2937'})
def col(i): return RAMP[min(i,len(RAMP)-1)]
def mark(i): return MARKS[min(i,len(MARKS)-1)]

# --- Figure 1 : la grille, résultat principal
def mean_by_size(sub):
    return (sub.groupby('context_size')
              .agg(x=('ctx','mean'),y=('gain_logloss','mean'),
                   sd=('gain_logloss','std'),n=('target','nunique')).sort_values('x'))

sub=FT[FT.phase.str.startswith('B')]
if len(sub):
    fig,ax=plt.subplots(figsize=(5.8,4.2))
    ax.set_xscale('log')
    ax.axhline(0,color=ORANGE,lw=2.0,zorder=2)
    for i,st in enumerate(sorted(sub.ft_steps.unique())):
        s_=sub[sub.ft_steps==st]
        g=mean_by_size(s_[s_.context_size!='full'])      # tailles fixes, comparables
        if not len(g): continue
        ax.errorbar(g.x,g.y,yerr=g.sd/np.sqrt(g.n.clip(lower=1)),color=col(i),lw=1.9,
                    marker=mark(i),ms=6,capsize=2.5,zorder=3)
        ax.annotate(f'{int(st)} pas',xy=(g.x.iloc[0],g.y.iloc[0]),xytext=(-7,0),
                    textcoords='offset points',color=col(i),fontsize=8,va='center',ha='right')
        f_=mean_by_size(s_[s_.context_size=='full'])      # abscisse = moyenne des trains
        if len(f_):
            ax.errorbar(f_.x,f_.y,yerr=f_.sd/np.sqrt(f_.n.clip(lower=1)),color=col(i),
                        marker='*',ms=13,ls='none',capsize=2.5,zorder=4,
                        label='tout le train (abscisse moyennée)' if i==0 else None)
    ax.axvline(REF_CONTEXT,color=GREY,lw=.8,ls='--',zorder=1)
    x0=0.35*float(sub.ctx.min()); x1=1.6*float(sub.ctx.max())   # place pour les étiquettes
    ax.set_xlim(x0,x1); y0,y1=ax.get_ylim()
    ax.annotate('modèle non adapté',xy=(x1,0),xytext=(-4,4),textcoords='offset points',
                color=ORANGE,fontsize=8,va='bottom',ha='right')
    ax.annotate(f'tes runs précédents\n({REF_CONTEXT} lignes)',xy=(REF_CONTEXT,y0),
                xytext=(-5,10),textcoords='offset points',color=GREY,fontsize=7.5,ha='right')
    ax.set_xlabel('Lignes dans le contexte d inférence (échelle log)')
    ax.set_ylabel('Gain de log-loss apporté par le fine-tuning')
    ax.set_title('Le fine-tuning est-il un substitut du contexte ?',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True)
    ax.legend(frameon=False,fontsize=7.5,loc='lower left')
    fig.tight_layout(); fig.savefig(ROOT/'fig1_grille.png',bbox_inches='tight'); plt.show()

# --- Figure 2 : la courbe de base, absolue
if len(B):
    fig,ax=plt.subplots(figsize=(5.6,4.0))
    for tid,g in B.groupby('target'):
        g=g.sort_values('ctx'); ax.plot(g.ctx,g.logloss,lw=1.0,color=BLUE,alpha=.30,
                                        marker='o',ms=2.5,mew=0,zorder=2)
    def mean_ll(sub):
        return (sub.groupby('context_size').agg(x=('ctx','mean'),y=('logloss','mean'),
                sd=('logloss','std'),n=('target','nunique')).sort_values('x'))
    m=mean_ll(B[B.context_size!='full'])        # tailles fixes, mêmes datasets partout
    ax.errorbar(m.x,m.y,yerr=m.sd/np.sqrt(m.n.clip(lower=1)),color=ORANGE,lw=2.3,marker='o',
                ms=7,capsize=3,zorder=4,label='moyenne, tailles fixes')
    mf=mean_ll(B[B.context_size=='full'])
    if len(mf):
        ax.errorbar(mf.x,mf.y,yerr=mf.sd/np.sqrt(mf.n.clip(lower=1)),color=ORANGE,marker='*',
                    ms=15,ls='none',capsize=3,zorder=5,label='tout le train (abscisse moyennée)')
    ax.plot([],[],lw=1.0,color=BLUE,alpha=.5,label='un dataset')
    ax.set_xscale('log'); ax.set_xlabel('Lignes dans le contexte (échelle log)')
    ax.set_ylabel('Log-loss sur le rapport scellé')
    ax.set_title('Ce que le contexte apporte, sans aucun entraînement',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=8)
    fig.tight_layout(); fig.savefig(ROOT/'fig2_courbe_base.png',bbox_inches='tight'); plt.show()

# --- Figure 3 : dispersion par dataset au budget maximal
s3=FT[(FT.phase.str.startswith('B'))&(FT.ft_steps==max(FT_MILESTONES))]
if len(s3):
    fig,ax=plt.subplots(figsize=(5.6,4.0))
    ax.axhline(0,color=ORANGE,lw=1.8,zorder=3)
    for tid,g in s3.groupby('target'):
        g=g.sort_values('ctx')
        ax.plot(g.ctx,g.gain_logloss,lw=1.0,color=RAMP[2],alpha=.30,marker='o',ms=2.5,
                mew=0,zorder=2)
    m3=s3.groupby('context_size').agg(x=('ctx','mean'),y=('gain_logloss','mean')).sort_values('x')
    ax.plot(m3.x,m3.y,lw=2.3,color=RAMP[2],marker='s',ms=7,zorder=4,label='moyenne')
    ax.plot([],[],lw=1.0,color=RAMP[2],alpha=.5,label='un dataset')
    ax.set_xscale('log'); ax.set_xlabel('Lignes dans le contexte (échelle log)')
    ax.set_ylabel('Gain de log-loss')
    ax.set_title(f'Dispersion entre datasets, {max(FT_MILESTONES)} pas de fine-tuning',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=8)
    fig.tight_layout(); fig.savefig(ROOT/'fig3_dispersion.png',bbox_inches='tight'); plt.show()

# --- Figure 4 : phase C
s4=FT[FT.target.isin(LONG_IDS)] if LONG_IDS else FT.iloc[0:0]
if len(s4[s4.phase.str.startswith('C')]):
    fig,ax=plt.subplots(figsize=(5.6,4.0))
    ax.axhline(0,color=ORANGE,lw=1.8,zorder=3)
    for j,(ph,st) in enumerate(sorted({(p,s) for p,s in zip(s4.phase,s4.ft_steps)})):
        g=(s4[(s4.phase==ph)&(s4.ft_steps==st)].groupby('context_size')
             .agg(x=('ctx','mean'),y=('gain_logloss','mean')).sort_values('x'))
        if len(g)<2: continue
        short=ph.startswith('B')
        ax.plot(g.x,g.y,lw=1.9,ls='-' if short else '--',color=col(0 if short else 2),
                marker=mark(j%3),ms=6,zorder=4,
                label=f'épisodes {FT_CONTEXT+FT_QUERY if short else LONG_FT_CONTEXT+LONG_FT_QUERY} l., {int(st)} pas')
    ax.set_xscale('log'); ax.set_xlabel('Lignes dans le contexte (échelle log)')
    ax.set_ylabel('Gain de log-loss')
    ax.set_title('La taille des épisodes déplace-t-elle la courbe ?',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=7.5)
    fig.tight_layout(); fig.savefig(ROOT/'fig4_episodes.png',bbox_inches='tight'); plt.show()

write_json(ROOT/'RUN_COMPLETE.json',dict(run_hash=RUN_HASH,
    datasets=int(CUR.target.nunique()),arms=sorted(CUR.arm.unique().tolist()),
    points=len(CUR),excluded=FAILURES,hours_left=round(hours_left(),2),
    note='development study; OpenML outer fold 0 never read'))
print('\nFichiers dans',ROOT)
for f in sorted(ROOT.glob('*.csv'))+sorted(ROOT.glob('*.png'))+sorted(ROOT.glob('*.json')):
    print('  ',f.name)

## Comment lire ce run**Le premier chiffre à regarder** est le gain moyen à 1024 lignes, au budget de 50 pas. C'est exactement le réglage de tes trois études précédentes. S'il ne retombe pas près de zéro comme elles, quelque chose a changé dans le harnais et rien d'autre n'est interprétable tant que ce n'est pas élucidé. C'est un contrôle de cohérence, pas un résultat.**Le deuxième** est la ligne à 32 lignes de contexte. C'est le régime le plus favorable qu'on puisse offrir au fine-tuning : le modèle n'a presque rien à lire et tout à retenir. Si le gain y est positif et significatif, l'article a sa thèse conditionnelle. S'il y est nul ou négatif, le négatif devient beaucoup plus solide qu'avant — et c'est alors ça, le résultat : *le fine-tuning de TabPFN 3.5 ne gagne à aucun budget de contexte, pas même quand le contexte est famélique.***Le troisième** est la table des pentes. Elle dit si la décroissance est un phénomène général ou l'effet de deux ou trois datasets. Un rho médian franchement négatif avec 18 ou 20 datasets sur 24 dans le même sens est une figure ; 13 sur 24 n'en est pas une, et il faut le dire.**Ce que la colonne `doublements_equivalents` apporte.** Elle traduit le fine-tuning dans l'unité que le lecteur comprend déjà, puisque ton run précédent a établi qu'un doublement de contexte vaut +0,0124 de log-loss. « 200 pas de fine-tuning valent −0,4 doublement de contexte » est une phrase qu'on retient, et qui tue le sujet proprement si le signe est négatif.**Ce que la phase C protège.** Sans elle, un relecteur attribuera toute la forme de la courbe à la taille des épisodes d'entraînement et l'article n'aura rien montré. Avec elle, soit la forme est stable et l'effet est bien celui du contexte d'inférence, soit elle se déplace et tu tiens une recette pratique. Si le budget de temps l'a sautée, elle mérite une session à elle seule avant d'écrire.**Avant de publier.** Trois graines. Le fine-tuning en a un besoin plus pressant que tes runs d'inférence : le tirage des épisodes est une source de variance réelle, et une pente estimée sur une seule graine se défend mal. Compte une session par graine, en changeant `SEED` et `RUN_TAG` ensemble.**Ce que ce notebook ne teste pas, et qu'il faut écrire dans les limites.** Un seul taux d'apprentissage, un seul optimiseur, tous les paramètres entraînables, et un fine-tuning épisodique sur la cible elle-même — pas de LoRA, pas de tête légère, pas d'adaptateur. Un résultat négatif ici porte sur *cette* famille de fine-tuning, pas sur toutes. Le dire d'avance vaut mieux que se le faire dire.